# 1. 환경 설정 및 필수 라이브러리 설치
BERT 모델과 데이터셋 핸들링, Excel 파일 로드를 위한 필수 패키지를 설치합니다.

In [ ]:
!pip install transformers datasets accelerate evaluate scikit-learn openpyxl

# 2. 데이터셋 로드 및 전처리
**중요:** 왼쪽 사이드바의 폴더 아이콘을 눌러 본인의 `.xlsx` 파일을 업로드한 뒤, 아래 `FILE_PATH`, `TEXT_COL`, `LABEL_COL`을 파일 상황에 맞게 수정하세요.

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import BertTokenizer

# ================= 설정 구역 =================
FILE_PATH = "FR_NFR_Dataset.xlsx"  # 업로드한 엑셀 파일명
TEXT_COL = "Requirement Text"                       # 요구사항 문장이 들어있는 컬럼명
LABEL_COL = "Type"                     # FR / NFR 레이블이 들어있는 컬럼명
BERT_MODEL = 'bert-base-uncased' # BERT 모델
# =============================================

# Excel 데이터 로드
df = pd.read_excel(FILE_PATH)

# 결측치 제거
df = df[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)

# 레이블이 문자열(예: 'FR', 'NFR')일 경우 자동 숫자로 매핑
if df[LABEL_COL].dtype == 'object':
    label_mapping = {val: idx for idx, val in enumerate(df[LABEL_COL].unique())}
    print(f"[안내] 텍스트 레이블을 숫자로 변환합니다: {label_mapping}")
    df[LABEL_COL] = df[LABEL_COL].map(label_mapping)
else:
    print("[안내] 레이블이 이미 숫자 형식입니다.")
    label_mapping = None

# 컬럼명 통일
df = df.rename(columns={TEXT_COL: 'text', LABEL_COL: 'label'})

# 학습/검증 데이터 분할 (층화 추출 적용)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Hugging Face 데이터셋 변환
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# 토크나이저 초기화 및 토크나이징
tokenizer = BertTokenizer.from_pretrained(BERT_MODEL)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
print("데이터 변환 및 토크나이징 완료!")

[안내] 텍스트 레이블을 숫자로 변환합니다: {'NFR': 0, 'FR': 1}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/4893 [00:00<?, ? examples/s]

Map:   0%|          | 0/1224 [00:00<?, ? examples/s]

데이터 변환 및 토크나이징 완료!


# 3. 모델 및 평가지표 정의
BERT 백본 위에 이진/다중 분류를 위한 Dense Layer(Linear)가 결합된 모델을 정의하고 GPU로 로드합니다.

In [ ]:
from transformers import BertForSequenceClassification
import evaluate

num_labels = df['label'].nunique()
model = BertForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=num_labels)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"모델이 다음 장치에 로드되었습니다: {device}")

# 평가지표(정확도 및 F1 스코어) 설정
clf_metrics = evaluate.combine(["accuracy", "f1"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return clf_metrics.compute(predictions=predictions, references=labels)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


모델이 다음 장치에 로드되었습니다: cuda


# 4. Trainer 설정 및 파인 튜닝 진행
Hugging Face Trainer 아규먼트를 세팅하고 학습을 수행합니다. (Colab T4 GPU 기준 약 2~5분 소요)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# 학습 시작
trainer.train()

print("\n=== 최종 검증 데이터셋 평가 결과 ===")
print(trainer.evaluate())

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.292952,0.358369,0.859477,0.885638
2,0.274663,0.330657,0.883170,0.907203
3,0.232447,0.353033,0.887255,0.911425


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte


=== 최종 검증 데이터셋 평가 결과 ===


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.232447,0.353033,3,0.887255,0.911425


{'eval_loss': 0.3530329763889313, 'eval_accuracy': 0.8872549019607843, 'eval_f1': 0.9114249037227214}


# 5. 실시간 요구사항 문장 분류 테스트
새로운 요구사항 문장을 입력하여 모델이 FR인지 NFR인지 올바르게 분류하는지 확인합니다.

In [ ]:
def predict_requirement(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_class = torch.argmax(probabilities, dim=-1).item()

    if label_mapping:
        inv_map = {v: k for k, v in label_mapping.items()}
        pred_label = inv_map[pred_class]
    else:
        pred_label = "NFR" if pred_class == 1 else "FR"

    print(f"문장: {text}")
    print(f"예측 결과: {pred_label} (확률: {probabilities[0][pred_class].item():.4f})\n")

# 테스트 예시
predict_requirement("The system shall allow administrators to revoke user access permissions immediately.")
predict_requirement("The system must be operational 99.9% of the time during business hours.")

문장: The system shall allow administrators to revoke user access permissions immediately.
예측 결과: FR (확률: 0.9899)

문장: The system must be operational 99.9% of the time during business hours.
예측 결과: NFR (확률: 0.9763)



# 6. 체크포인트 저장
구글 드라이브에 저장한다.

In [ ]:
# 1. 구글 드라이브 마운트 (연결)
from google.colab import drive
drive.mount('/content/drive')

# 2. 내 구글 드라이브의 특정 폴더에 모델 저장
# (드라이브 내에 'bert_model'이라는 폴더가 자동 생성되며 저장됩니다)
save_path = "/content/drive/MyDrive/bert_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"구글 드라이브의 {save_path} 경로에 안전하게 저장 완료되었습니다!")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

구글 드라이브의 /content/drive/MyDrive/bert_model 경로에 안전하게 저장 완료되었습니다!


# 7. 실사용
직접 사용해본다.

In [2]:
# 1. 구글 드라이브 마운트 (연결)
# 실행 후 나타나는 팝업창에서 구글 계정 로그인을 완료해주세요.
from google.colab import drive
drive.mount('/content/drive')

# 2. 필수 라이브러리 설치
!pip install transformers accelerate

import torch
from transformers import BertTokenizer, BertForSequenceClassification

# 3. 구글 드라이브에 저장했던 모델 경로 설정
# (이전 단계에서 모델을 저장했던 구글 드라이브 폴더 경로와 일치해야 합니다)
MODEL_PATH = "/content/drive/MyDrive/bert_model"

# 4. 저장된 토크나이저와 모델 불러오기
print("구글 드라이브에서 학습된 모델을 로드하는 중입니다...")
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model = BertForSequenceClassification.from_pretrained(MODEL_PATH)

# GPU가 사용 가능하면 GPU로, 아니면 CPU로 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ⚠️ 중요: 모델을 '평가/추론 모드'로 설정 (Dropout 등의 레이어가 비활성화됩니다)
model.eval()
print(f"✅ 모델 로드 완료! (사용 장치: {device})")

# 5. 요구사항 분류(추론) 함수 정의
def predict_requirement(text, ans):
    """
    영어 요구사항 문장을 입력받아 FR 또는 NFR로 예측합니다.
    """
    # 입력 문장 토크나이징 및 텐서 변환 후 GPU/CPU로 이동
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    # 기울기(Gradient) 계산을 비활성화하여 메모리 절약 및 속도 향상
    with torch.no_grad():
        outputs = model(**inputs)

    # 모델의 출력값(Logits)을 Softmax를 통해 0~1 사이의 확률값으로 변환
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)

    # 가장 높은 확률을 가진 클래스의 인덱스 추출
    pred_class = torch.argmax(probabilities, dim=-1).item()

    # 레이블
    class_labels = {
        0: "Non-Functional Requirement (NFR)",
        1: "Functional Requirement (FR)"
    }

    predicted_label = class_labels.get(pred_class, f"Class {pred_class}")
    confidence = probabilities[0][pred_class].item()
    okay = pred_class == ans

    print(f"[-] 입력 문장: {text}")
    print(f"[+] 예측 결과: {predicted_label} (확률: {confidence:.4f}) ({"정답" if okay else "오답"})\n")

    return 1 if okay else 0

# ==========================================================
# 6. 실제 사용 테스트 (새로운 요구사항 문장 입력)
# ==========================================================

print("\n=== 🔍 실시간 요구사항 분류 테스트 ===")

sen = {
  "FR": [
    "The system shall allow users to browse toys by category.",
    "The system shall allow users to search for toys by name or keyword.",
    "The system shall display detailed information for each toy product including name, price, description, and image.",
    "The system shall allow users to add toys to a shopping cart.",
    "The system shall allow users to modify quantities of items in the shopping cart.",
    "The system shall allow users to remove items from the shopping cart.",
    "The system shall calculate and display the total price of items in the shopping cart.",
    "The system shall allow users to register as a member with email and password.",
    "The system shall allow registered users to log in and log out.",
    "The system shall allow users to place an order with shipping address and payment information.",
    "The system shall send an order confirmation email to the user after successful order placement.",
    "The system shall allow users to view their order history.",
    "The system shall allow users to add toys to a wishlist.",
    "The system shall allow users to write and view product reviews.",
    "The system shall display recommended toys based on user preferences or best sellers.",
  ],
  "NFR": [
    "The website shall be accessible on desktop and mobile browsers.",
    "The website shall support Chrome, Firefox, Safari, and Edge browsers.",
    "The product page shall load within 3 seconds under normal network conditions.",
    "All payment transactions shall be processed through a secure payment gateway.",
    "User passwords shall be stored using secure hashing algorithms.",
    "The website shall comply with applicable data protection regulations.",
    "The website shall maintain responsiveness across screen sizes from 320px to 1920px width.",
    "The system shall support concurrent access for at least 100 users simultaneously."
  ]
}

okay_fr = 0
okay_nfr = 0

for s in sen["FR"]:
  okay_fr += predict_requirement(s, 1)

for s in sen["NFR"]:
  okay_nfr += predict_requirement(s, 0)

print(f"FR 정답률 {(okay_fr) / (len(sen["FR"])):.4f}")
print(f"NFR 정답률 {(okay_nfr) / (len(sen["NFR"])):.4f}")
print(f"전체 정답률 {(okay_fr + okay_nfr) / (len(sen["FR"]) + len(sen["NFR"])):.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
구글 드라이브에서 학습된 모델을 로드하는 중입니다...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 모델 로드 완료! (사용 장치: cuda)

=== 🔍 실시간 요구사항 분류 테스트 ===
[-] 입력 문장: The system shall allow users to browse toys by category.
[+] 예측 결과: Functional Requirement (FR) (확률: 0.9880) (정답)

[-] 입력 문장: The system shall allow users to search for toys by name or keyword.
[+] 예측 결과: Functional Requirement (FR) (확률: 0.9904) (정답)

[-] 입력 문장: The system shall display detailed information for each toy product including name, price, description, and image.
[+] 예측 결과: Functional Requirement (FR) (확률: 0.9891) (정답)

[-] 입력 문장: The system shall allow users to add toys to a shopping cart.
[+] 예측 결과: Functional Requirement (FR) (확률: 0.9891) (정답)

[-] 입력 문장: The system shall allow users to modify quantities of items in the shopping cart.
[+] 예측 결과: Functional Requirement (FR) (확률: 0.9895) (정답)

[-] 입력 문장: The system shall allow users to remove items from the shopping cart.
[+] 예측 결과: Functional Requirement (FR) (확률: 0.9888) (정답)

[-] 입력 문장: The system shall calculate and display the total price of items in the s